## Code Generator 

#### Goal - To generate High Performance C++ from python code

In [34]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:7]}")
else:
    print("Deepseek API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")


In [36]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
deepseek_url = "https://api.deepseek.com/v1/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)

deepseek = OpenAI(base_url=deepseek_url, api_key=deepseek_api_key)

grok = OpenAI(base_url=grok_url, api_key=grok_api_key)



In [37]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-6"
GROK_MODEL = "grok-4"
DEEPSEEK_MODEL = "deepseek-reasoner"

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

In [40]:
compile_command = ["clang++", "-std=c++20", "-O3", "-flto=thin", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [62]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
Do NOT use #include <bits/stdc++.h> — use only standard headers compatible with Apple Clang.
"""

def user_prompt(python_code):
    return f"""
    Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {system_info}
    Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
    {compile_command}
    Respond only with C++ code.
    Python code to port:

    ```python
    {python_code}
    ```
    """

In [42]:
def message_for(python_code):
    return [
        {"role":"system", "content":system_prompt},
        {"role" : "user", "content":user_prompt(python_code)}
    ]

In [43]:
def write_output(cpp_code):
    with open("main.cpp" , "w" , encoding = "utf-8") as f:
        f.write(cpp_code)

In [53]:
def port(client, model, python_code):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model = model, messages = message_for(python_code), reasoning_effort =  reasoning_effort)
    res = response.choices[0].message.content
    res = res.replace('```cpp', '').replace('```', "")
    write_output(res)

In [45]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [46]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

In [48]:
port(openai, OPENAI_MODEL, pi)

In [49]:
import subprocess
def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
#anthropic

port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(deepseek, DEEPSEEK_MODEL, pi)
compile_and_run()